In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import animation
import glide.calibration.radiation as rad

def get_pixels():
    """Get all pixel coordinates and return in flattened u,v array"""
    uu = np.linspace(0.5, 512 - .5, num=512)
    vv = np.linspace(0.5, 512 - .5, num=512)
    u_pix, v_pix = np.meshgrid(uu, vv)
    uv = np.stack((np.ravel(u_pix), np.ravel(v_pix)), axis=0)
    return uv
def gen_fov_mask(radii):
    """Create FOV circular mask for WFI. radii in pixels. Everything inside radii is 1, else 0"""
    ctr = 512 / 2

    # Calculate radial angle of each pixel
    uv = get_pixels()
    u_pix = uv[0, :] - ctr
    v_pix = uv[1, :] - ctr
    rpix = np.sqrt(u_pix**2 + v_pix**2)
    rpix = np.reshape(rpix, (512,512))

    # Construct mask
    rmask = np.zeros(np.shape(rpix))
    rmask[rpix <= radii] = 1

    return rmask

filename = 'data/WFI_1A-DRK/CARRUTHERS_GCI-WFI_L1A-DRK_20260119_v1.0.nc'
ds = xr.open_dataset(filename)
ims = ds["images"]

# test imshow with data
plt.imshow(ims[0,:,:], vmin=0, vmax=250000/50)

#generate fov mask, 260 pixels is entirely within FOV, 300 pixels intrudes into all corners
mask_fov = gen_fov_mask(260).astype(bool)
mask_cnr = ~gen_fov_mask(300).astype(bool)
t_int = ds["t_int"].values
image = ims[0,:,:]

# TESTING: Create a colored array for the overlay (e.g., red color)
overlay_color1 = np.zeros((*image.shape, 4)) # RGBA
overlay_color1[mask_fov] = [1, 0, 0, 1] # Red (R, G, B, A) - full opacity
overlay_color2 = np.zeros((*image.shape, 4)) # RGBA
overlay_color2[mask_cnr] = [0, 1, 0, 1] # Red (R, G, B, A) - full opacity

# Display the original image
plt.imshow(image, vmin=0, vmax=250000/40)

# Overlay the mask using imshow with an alpha value
plt.imshow(overlay_color1, alpha=0.5)
plt.imshow(overlay_color2, alpha=0.5)

plt.title('Image with FOV & Corner Mask Overlays')
plt.show()

# run SEP factor retrival
aps_rad, mcp_rad, scaling_factor, mcp_gain = rad.retrieve_radiation(ims, mask_fov, mask_cnr, t_int)

print(aps_rad, mcp_rad, scaling_factor, mcp_gain)

In [ ]:
# First set up the figure, the axis, and the plot element we want to animate
fig = plt.figure()
ax = plt.axes(xlim=(0, 512), ylim=(0, 512))
a=ims[0,:,:]
im=plt.imshow(a,interpolation='none', vmin=0, vmax=250000/10)

# initialization function: plot the background of each frame
def init():
    im.set_data(ims[0,:,:])
    return [im]

# animation function.  This is called sequentially
def animate(i):
    a=im.get_array()
    a=ims[i,:,:]    # update values with next frame data
    im.set_array(a)
    return [im]

# call the animator.  blit=True means only re-draw the parts that have changed.
anim = animation.FuncAnimation(fig, animate, init_func=init,
                               frames=16, interval=10000000000000000, blit=True)

# save the animation as an mp4.  This requires ffmpeg or mencoder to be
# installed.  The extra_args ensure that the x264 codec is used, so that
# the video can be embedded in html5.  You may need to adjust this for
# your system: for more information, see
# http://matplotlib.sourceforge.net/api/animation_api.html
anim.save('basic_animation.mp4', fps=1, extra_args=['-vcodec', 'libx264'])


In [ ]:
"""
Matplotlib Animation Example

author: Jake Vanderplas
email: vanderplas@astro.washington.edu
website: http://jakevdp.github.com
license: BSD
Please feel free to use and modify this, but keep the above information. Thanks!
"""

import numpy as np
from matplotlib import pyplot as plt
from matplotlib import animation

# First set up the figure, the axis, and the plot element we want to animate
fig = plt.figure()
ax = plt.axes(xlim=(0, 2), ylim=(-2, 2))
line, = ax.plot([], [], lw=2)

# initialization function: plot the background of each frame
def init():
    line.set_data([], [])
    return line,

# animation function.  This is called sequentially
def animate(i):
    x = np.linspace(0, 2, 30)
    y = np.sin(2 * np.pi * (x - 0.01 * i))
    line.set_data(x, y)
    return line,

# call the animator.  blit=True means only re-draw the parts that have changed.
anim = animation.FuncAnimation(fig, animate, init_func=init,
                               frames=200, interval=20, blit=True)

# save the animation as an mp4.  This requires ffmpeg or mencoder to be
# installed.  The extra_args ensure that the x264 codec is used, so that
# the video can be embedded in html5.  You may need to adjust this for
# your system: for more information, see
# http://matplotlib.sourceforge.net/api/animation_api.html
anim.save('basic_animation.mp4', fps=30, extra_args=['-vcodec', 'libx264'])

plt.show()